# Baghdad_gym_dataset.csv: Data Boss session

Exported from Data Boss. Run the cells from top to bottom next to the data file to repeat every cleaning step and chart. Each step has the message the app showed when it was applied.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## Load the data

Reads `Baghdad_gym_dataset.csv` exactly as Data Boss did.

In [ ]:
df = pd.read_csv('Baghdad_gym_dataset.csv')
df.shape

## Cleaning steps

### Step 1

Dropped 5 column(s): state, countryCode, categories/1, categories/2, categoryName.

Rows × columns: 100 × 14 → 100 × 9

In [ ]:
df = df.drop(columns=['state', 'countryCode', 'categories/1', 'categories/2', 'categoryName'])
df.shape

### Step 2

Replaced 1 occurrence(s) of 'الكمالية،' with 'Baghdad' in 'city'.

Rows × columns: 100 × 9 → 100 × 9

In [ ]:
df['city'] = df['city'].replace('الكمالية،', 'Baghdad')
df.shape

### Step 3

Dropped 10 rows with missing values in totalScore.

Rows × columns: 100 × 9 → 90 × 9

In [ ]:
df = df.dropna(subset=['totalScore'])
df.shape

### Step 4

Added column 'has_website' = CASE WHEN website IS NULL THEN 'no' ELSE 'yes' END.

Rows × columns: 90 × 9 → 90 × 10

In [ ]:
import duckdb
con = duckdb.connect()
con.register('data', df.assign(__databoss_row=range(len(df))))
df['has_website'] = con.sql("""SELECT (CASE WHEN website IS NULL THEN 'no' ELSE 'yes' END) AS v FROM data ORDER BY __databoss_row""").df()['v'].to_numpy()
df.shape

### Step 5

Added column 'site_type' = CASE WHEN website IS NULL THEN 'none' WHEN website ILIKE '%facebook%' OR website ILIKE '%instagram%' THEN 'social page' ELSE 'own website' END.

Rows × columns: 90 × 10 → 90 × 11

In [ ]:
import duckdb
con = duckdb.connect()
con.register('data', df.assign(__databoss_row=range(len(df))))
df['site_type'] = con.sql("""SELECT (CASE WHEN website IS NULL THEN 'none' WHEN website ILIKE '%facebook%' OR website ILIKE '%instagram%' THEN 'social page' ELSE 'own website' END) AS v FROM data ORDER BY __databoss_row""").df()['v'].to_numpy()
df.shape

### Step 6

Added column 'has_phone' = CASE WHEN phone IS NULL THEN 'no' ELSE 'yes' END.

Rows × columns: 90 × 11 → 90 × 12

In [ ]:
import duckdb
con = duckdb.connect()
con.register('data', df.assign(__databoss_row=range(len(df))))
df['has_phone'] = con.sql("""SELECT (CASE WHEN phone IS NULL THEN 'no' ELSE 'yes' END) AS v FROM data ORDER BY __databoss_row""").df()['v'].to_numpy()
df.shape

### Step 7

Binned reviewsCount into 4 custom ranges as review_band.

Rows × columns: 90 × 12 → 90 × 13

In [ ]:
df['review_band'] = pd.cut(df['reviewsCount'], bins=[0.0, 5.0, 20.0, 100.0, float('inf')], labels=['0–5', '5–20', '20–100', '> 100'], include_lowest=True)
df.shape

### Step 8

Added column 'adjusted_rating' = ROUND((totalScore * reviewsCount + 4.31 * 12) / (reviewsCount + 12), 2).

Rows × columns: 90 × 13 → 90 × 14

In [ ]:
import duckdb
con = duckdb.connect()
con.register('data', df.assign(__databoss_row=range(len(df))))
df['adjusted_rating'] = con.sql("""SELECT (ROUND((totalScore * reviewsCount + 4.31 * 12) / (reviewsCount + 12), 2)) AS v FROM data ORDER BY __databoss_row""").df()['v'].to_numpy()
df.shape

## Result

In [ ]:
df = df.reset_index(drop=True)
df.head(10)

## Charts

Drawn from the cleaned data above.

### HIST: {"column":"totalScore","bins":"12"}

- 90 values. Mean 4.31, median 4.4, standard deviation 0.525.
- The middle 50% of values lie between 4.03 and 4.67.
- The distribution is left-skewed: most values are high with a tail of low values (skewness -1.20).
- With this much skew, the median is a better “typical value” than the mean.

In [ ]:
fig, ax = plt.subplots(figsize=(9.0, 5.0))
sns.histplot(data=df, x='totalScore', bins=12, kde=True, ax=ax)
ax.axvline(df['totalScore'].median(), linestyle='--', linewidth=1)
plt.tight_layout()
plt.show()

### BOX: {"y":"totalScore","x":"review_band"}

- Highest median: 0–5 (4.5). Lowest: 20–100 (4.2).
- Overall median 4.4; middle 50% between 4.03 and 4.67.
- 4 point(s) fall outside the whiskers (1.5 × IQR).

In [ ]:
fig, ax = plt.subplots(figsize=(9.0, 5.0))
order = [c for c in df['review_band'].cat.categories if c in set(df['review_band'].dropna())][:12]
sns.boxplot(data=df[df['review_band'].isin(order)], x='review_band', y='totalScore', order=order, ax=ax)
plt.tight_layout()
plt.show()

### SCAT: {"x":"reviewsCount","y":"totalScore","log_x":true,"trend":false}

- Pearson r = -0.05: no clear linear relationship (90 points).
- Rank correlation (Spearman) is -0.27, so the relationship is not a straight line or is driven by a few extreme points.

In [ ]:
fig, ax = plt.subplots(figsize=(9.0, 5.0))
sns.scatterplot(data=df, x='reviewsCount', y='totalScore', alpha=0.85, s=40, ax=ax)
ax.set_xscale('log')
plt.tight_layout()
plt.show()

### BAR: {"x":"site_type","y":"totalScore","agg":"mean"}

- Highest: own website (4.69). Lowest: social page (4.21).
- The highest group is 1.1× the lowest.

In [ ]:
s = df.groupby('site_type')['totalScore'].mean().sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(9.0, 5.0))
ax.bar(s.index.astype(str), s.values)
plt.tight_layout()
plt.show()

## Save the cleaned data

In [ ]:
df.to_csv('Baghdad_gym_dataset_cleaned.csv', index=False)